# Silver IAM/IGA — validação ponta a ponta

Execute com o kernel Python do projeto, em ordem. Usa os sete arquivos/fontes reais, sem gabarito. Cria um warehouse temporário separado e valida a reexecução. Não usa nem limpa o warehouse do projeto. O fluxo pode levar alguns minutos. Pare outros kernels Spark antes de iniciar.

In [ ]:
import sys
from pathlib import Path

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Projeto:", ROOT)

In [ ]:
import os
import tempfile

from pyspark.sql import SparkSession

from sod_platform.bronze.ingestion.spark import create_spark_session

assert SparkSession.getActiveSession() is None, (
    "Reinicie o kernel antes de executar este notebook."
)
previous_warehouse = os.environ.get("SOD_WAREHOUSE")
workspace = tempfile.TemporaryDirectory(prefix="sod-silver-notebook-")
os.environ["SOD_WAREHOUSE"] = str(Path(workspace.name) / "warehouse")
os.environ.setdefault("SPARK_MASTER", "local[2]")
spark = create_spark_session(ROOT, "silver-notebook")
print("Warehouse isolado:", os.environ["SOD_WAREHOUSE"])

In [ ]:
from scripts.validate_silver import validate

report = validate(spark, ROOT)
assert report["silver_first"]["run_status"] == "success"
assert report["silver_replay"]["run_status"] == "success"
print("Contagens Silver:", report["silver_first"]["rows_written"])
print("Quarentena:", report["silver_first"]["quarantined"])
print("Requests por status:", report["request_statuses"])

## Evidências de qualidade e contexto

O contexto possui uma linha por identidade × entitlement. Contas ficam em `accounts`, com cardinalidade explícita; a última recertificação observada até 2025-02-01 aparece com prefixo `certification_`. `approval_exists` é evidência documental, não decisão de legitimidade. Não há expiração presumida.

In [ ]:
spark.sql("SHOW TABLES IN sod.silver").show(truncate=False)
spark.sql(
    "SELECT error_code, count(*) AS registros FROM sod.quality.quarantine GROUP BY error_code ORDER BY registros DESC"
).show(truncate=False)
spark.sql(
    "SELECT approval_exists, never_used, certification_exists, count(*) AS acessos FROM sod.silver.access_context GROUP BY approval_exists, never_used, certification_exists"
).show()
spark.sql(
    "SELECT status, records_in, records_written, records_quarantined, duration_seconds FROM sod.metadata.silver_runs ORDER BY started_at"
).show()

In [ ]:
import json

output = ROOT / "docs/silver-validation.json"
output.write_text(json.dumps(report, indent=2))
print("Relatório sem payloads pessoais:", output)

## Encerrar

A célula abaixo remove apenas o warehouse temporário deste notebook. Se a execução falhar, ainda execute esta célula para liberar a sessão.

In [ ]:
spark.stop()
workspace.cleanup()
if previous_warehouse is None:
    os.environ.pop("SOD_WAREHOUSE", None)
else:
    os.environ["SOD_WAREHOUSE"] = previous_warehouse
print("Sessão encerrada e warehouse temporário removido.")